# Étape 5 — v3 : entraînement sur les étiquettes réannotées

Changements par rapport à la v2 :
1. **Étiquettes v3** (`train.jsonl` / `test.jsonl` réannotés selon `GUIDE_ANNOTATION.md`).
2. **Règles du prompt système** alignées sur le guide (test « rubrique Compétences d'un CV », noms canoniques, séparation des composés).
3. **Métriques complètes** : strictes, `poste_souple`, compétences souples et **taux d'ancrage** (mot entier).
4. **Base, v1 et v2 sont rescorées avec exactement le même code**, à partir de leurs sorties déjà enregistrées : tout le tableau est comparable.

Même découpage, mêmes hyperparamètres et même ordre des champs que la v2.

**Avant de lancer :** Exécution → Modifier le type d'exécution → GPU T4. Durée : 35 à 50 min.

In [ ]:
# 1. Installation + Google Drive
!pip install -q transformers accelerate peft bitsandbytes datasets pydantic pandas

from google.colab import drive
drive.mount('/content/drive')

import os
DOSSIER = '/content/drive/MyDrive/job-offer-extractor/data'
RESULTATS = '/content/drive/MyDrive/job-offer-extractor/resultats'
TRAIN = f'{DOSSIER}/train.jsonl'
TEST = f'{DOSSIER}/test.jsonl'
assert os.path.exists(TEST), 'test.jsonl introuvable : lance d’abord l’étape 3'

In [ ]:
# 2. Connexion Hugging Face
from huggingface_hub import login
login()

In [ ]:
# 3. Schéma avec `competences` EN DERNIER
import json, re
from typing import Literal, Optional
from pydantic import BaseModel, ValidationError

class Offre(BaseModel):
    poste: str
    entreprise: Optional[str]
    ville: Optional[str]
    type_contrat: Optional[Literal['PFE', 'stage', 'CDI', 'CDD', 'SIVP', 'freelance']]
    mode_travail: Optional[Literal['présentiel', 'hybride', 'remote']]
    duree_mois: Optional[float]
    remuneration: Optional[str]
    langue_offre: Literal['fr', 'en', 'ar']
    competences: list[str]

ORDRE = list(Offre.model_fields)
SCHEMA = json.dumps(Offre.model_json_schema(), ensure_ascii=False)

REGLES = '''Règles :
- poste = le titre de l'offre, sans "(H/F)" ni "F/H".
- type_contrat = "PFE" si l'offre parle de PFE ou de fin d'études, sinon "stage" pour un stage. Si plusieurs types sont cités, prendre le premier.
- remuneration = le montant s'il est donné ; "rémunéré" si l'offre dit seulement rémunéré ; une prime n'est pas un salaire ; "Confidentiel" = null.
- competences = tout ce qui s'écrirait dans la rubrique Compétences d'un CV : langages, frameworks, outils, logiciels, plateformes cloud, bases de données, méthodes, domaines techniques. Inclure aussi celles citées en exemple ou entre parenthèses.
- Ne sont PAS des compétences : tâches, concepts génériques (API seule, cache, scripts), secteurs d'activité, années d'expérience, diplômes, qualités humaines, noms d'entreprise seuls, technologies citées seulement dans la présentation de l'entreprise.
- Noms canoniques en alphabet latin, quelle que soit la langue de l'offre (Python, JavaScript). Séparer les compétences composées (Git/GitLab -> Git, GitLab), sauf CI/CD, TCP/IP, PL/SQL, Pub/Sub. "IA" s'écrit "intelligence artificielle". Chaque compétence apparaît une seule fois.
- langue_offre = UNE seule valeur, la langue dominante.
- ville et entreprise sont deux champs séparés.
- Si une information est absente, mets null. Ne devine jamais.'''

SYSTEM_PROMPT = (
    "Tu extrais les informations d'une offre d'emploi ou de stage. "
    'Réponds UNIQUEMENT avec un JSON valide respectant ce schéma :\n'
    + SCHEMA + '\n\n' + REGLES
)
print('Ordre des champs :', ORDRE)

In [ ]:
# 4. Charger le découpage existant (le même que la v1)
from collections import Counter

def lire(chemin):
    with open(chemin, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

train, test = lire(TRAIN), lire(TEST)
print(f'Train : {len(train)} | Test : {len(test)}')
print('Test, langues :', dict(Counter(d['extraction']['langue_offre'] for d in test)))

In [ ]:
# 5. Charger le modèle de base en 4 bits
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = 'Qwen/Qwen2.5-1.5B-Instruct'
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
tokenizer.padding_side = 'right'

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, quantization_config=bnb, device_map='auto')

In [ ]:
# 6. Exemples d'entraînement avec les champs dans le nouvel ordre
from datasets import Dataset

MAX_LEN = 2560

def construire(ex):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': ex['offre']}]
    prompt_ids = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, return_dict=True
    )['input_ids']
    extraction = {k: ex['extraction'][k] for k in ORDRE}
    reponse = json.dumps(extraction, ensure_ascii=False) + tokenizer.eos_token
    reponse_ids = tokenizer(reponse, add_special_tokens=False)['input_ids']
    input_ids = (prompt_ids + reponse_ids)[:MAX_LEN]
    labels = ([-100] * len(prompt_ids) + reponse_ids)[:MAX_LEN]
    return {'input_ids': input_ids, 'attention_mask': [1] * len(input_ids), 'labels': labels}

train_ds = Dataset.from_list([construire(ex) for ex in train])
print(len(train_ds), 'exemples | longueur max', max(len(x) for x in train_ds['input_ids']))
print('Exemple de cible :', json.dumps({k: train[0]['extraction'][k] for k in ORDRE}, ensure_ascii=False)[:200], '…')

In [ ]:
# 7. LoRA + entraînement (mêmes hyperparamètres que la v1)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import Trainer, TrainingArguments, DataCollatorForSeq2Seq

model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias='none', task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
))

args = TrainingArguments(
    output_dir='/content/qlora-offres-v3',
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    lr_scheduler_type='cosine',
    warmup_steps=2,
    fp16=True,
    optim='paged_adamw_8bit',
    gradient_checkpointing=True,
    logging_steps=5,
    save_strategy='no',
    report_to='none',
    seed=42,
)
trainer = Trainer(
    model=model, args=args, train_dataset=train_ds,
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
)
model.config.use_cache = False
trainer.train()
model.config.use_cache = True

In [ ]:
# 8. Sauvegarder tout de suite (avant l'évaluation, au cas où Colab se déconnecte)
ADAPTER = '/content/drive/MyDrive/job-offer-extractor/adapter_v3'
model.save_pretrained(ADAPTER)
tokenizer.save_pretrained(ADAPTER)
print('Adaptateur v3 sauvegardé dans', ADAPTER)

## Évaluation : génération une seule fois, puis scores brut et réparé

On génère les 14 sorties une fois (environ 12 min), puis on calcule deux scores sur **les mêmes sorties** :
- **brut** : la sortie telle quelle ;
- **réparé** : si le JSON est invalide, la liste des compétences est coupée au premier doublon et le JSON est refermé.

In [ ]:
# 9. Génération, réparation et scores
import pandas as pd

def nettoyer(texte):
    return texte.strip().removeprefix('```json').removesuffix('```').strip()

@torch.no_grad()
def extraire(offre, max_new_tokens=900):
    messages = [{'role': 'system', 'content': SYSTEM_PROMPT},
                {'role': 'user', 'content': offre}]
    inputs = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, return_tensors='pt', return_dict=True
    ).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                         pad_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)

def reparer(brut):
    """Coupe la liste des compétences au premier doublon et referme le JSON.
    Fonctionne parce que `competences` est le dernier champ."""
    t = nettoyer(brut)
    i = t.find('"competences"')
    if i < 0:
        return t
    debut = t.index('[', i)
    items, vus = [], set()
    for m in re.finditer(r'"((?:[^"\\]|\\.)*)"', t[debut:]):
        try:
            item = json.loads('"' + m.group(1) + '"')
        except json.JSONDecodeError:
            break
        if item.strip().lower() in vus:
            break
        vus.add(item.strip().lower())
        items.append(item)
    return t[:debut] + json.dumps(items, ensure_ascii=False) + '}'

CHAMPS = ['poste', 'entreprise', 'ville', 'type_contrat', 'mode_travail',
          'duree_mois', 'remuneration', 'langue_offre']

def norm(v):
    if v is None:
        return None
    if isinstance(v, (int, float)):
        return round(float(v), 2)
    return ' '.join(str(v).lower().split())

def egal(champ, pred, vrai):
    if champ == 'duree_mois' and pred is not None and vrai is not None:
        try:
            return abs(float(pred) - float(vrai)) < 0.05
        except (TypeError, ValueError):
            return False
    return norm(pred) == norm(vrai)

GARDER_SLASH = {'ci/cd', 'tcp/ip', 'pl/sql', 'pub/sub', 'tests a/b', 'a/b'}

def norm_souple(v):
    if v is None:
        return None
    t = str(v).lower()
    t = re.sub(r'\(?\b(h/f|f/h)\b\)?', ' ', t)
    t = re.sub(r'[^\w\s#+./]', ' ', t)
    return ' '.join(t.split())

def eclater(competences):
    """Normalise et sépare sur « / », sauf CI/CD, TCP/IP, etc."""
    out = set()
    for c in competences or []:
        if not isinstance(c, str):
            continue
        n = norm_souple(c)
        if n in GARDER_SLASH:
            out.add(n)
        else:
            out.update(p.strip() for p in n.split('/') if p.strip())
    return out

def ancre(terme, texte):
    """Le terme apparaît-il dans l'offre comme mot entier ?"""
    motif = r'(?<!\w)' + re.escape(terme.strip()) + r'(?!\w)'
    return re.search(motif, texte, flags=re.IGNORECASE) is not None

def scorer(nom, sorties, reparation):
    lignes, tp, fp, fn = [], 0, 0, 0
    stp, sfp, sfn, n_pred, n_ancre = 0, 0, 0, 0, 0
    for ex, brut in zip(test, sorties):
        vrai = ex['extraction']
        ligne = {'url': ex.get('url'), 'langue': vrai['langue_offre'], 'sortie_brute': brut,
                 'repare': False}
        try:
            pred = json.loads(nettoyer(brut))
            ligne['json_valide'] = True
        except json.JSONDecodeError:
            ligne['json_valide'] = False
            pred = {}
            if reparation:
                try:
                    pred = json.loads(reparer(brut))
                    ligne['repare'] = True
                except (json.JSONDecodeError, ValueError):
                    pred = {}
        try:
            Offre.model_validate(pred)
            ligne['schema_ok'] = True
        except ValidationError:
            ligne['schema_ok'] = False
        for c in CHAMPS:
            ligne[c] = egal(c, pred.get(c), vrai.get(c))
        ligne['poste_souple'] = norm_souple(pred.get('poste')) == norm_souple(vrai.get('poste'))
        comps_pred = [x for x in (pred.get('competences') or []) if isinstance(x, str)]
        p = {norm(x) for x in comps_pred}
        v = {norm(x) for x in vrai['competences']}
        tp += len(p & v); fp += len(p - v); fn += len(v - p)
        ps, vs = eclater(comps_pred), eclater(vrai['competences'])
        stp += len(ps & vs); sfp += len(ps - vs); sfn += len(vs - ps)
        morceaux = [m.strip() for x in comps_pred
                    for m in ([x] if norm_souple(x) in GARDER_SLASH else x.split('/')) if m.strip()]
        n_pred += len(morceaux)
        n_ancre += sum(ancre(m, ex['offre']) for m in morceaux)
        lignes.append(ligne)
    df = pd.DataFrame(lignes)

    def prf(a, b, c):
        pr = a / (a + b) if a + b else 0
        rp = a / (a + c) if a + c else 0
        return pr, rp, (2 * pr * rp / (pr + rp) if pr + rp else 0)

    prec, rapp, f1 = prf(tp, fp, fn)
    sprec, srapp, sf1 = prf(stp, sfp, sfn)
    resume = {'JSON valide': df['json_valide'].mean(),
              'Réparées': df['repare'].mean(),
              'Schéma respecté': df['schema_ok'].mean()}
    resume.update({c: df[c].mean() for c in CHAMPS})
    resume['poste_souple'] = df['poste_souple'].mean()
    resume.update({'Compétences précision': prec, 'Compétences rappel': rapp, 'Compétences F1': f1,
                   'Compétences souples P': sprec, 'Compétences souples R': srapp,
                   'Compétences souples F1': sf1,
                   'Taux d’ancrage': n_ancre / n_pred if n_pred else 0,
                   'Compétences / offre': n_pred / len(df)})
    df.to_csv(f'{RESULTATS}/details_{nom}.csv', index=False)
    return pd.Series(resume, name=nom).round(3)

model.eval()
sorties = []
for i, ex in enumerate(test):
    sorties.append(extraire(ex['offre']))
    print(f'{i+1}/{len(test)}', end='\r')

res_v3_brut = scorer('finetune_v3_brut', sorties, reparation=False)
res_v3_repare = scorer('finetune_v3_repare', sorties, reparation=True)

In [ ]:
# 10. Rescorer base, v1 et v2 avec le même code et les étiquettes v3, puis tableau final
def sorties_de(nom):
    d = pd.read_csv(f'{RESULTATS}/details_{nom}.csv')
    assert list(d['url']) == [ex.get('url') for ex in test], f'Ordre différent pour {nom}'
    return d['sortie_brute'].fillna('').tolist()

res_base = scorer('base_rescore_v3', sorties_de('base'), reparation=False)
res_v1 = scorer('finetune_v1_rescore_v3', sorties_de('finetune_v1'), reparation=False)
res_v2 = scorer('finetune_v2_rescore_v3', sorties_de('finetune_v2_brut'), reparation=False)

final = pd.concat([res_base.rename('base'), res_v1.rename('v1'), res_v2.rename('v2'),
                   res_v3_brut.rename('v3 brut'), res_v3_repare.rename('v3 réparé')], axis=1)
final.to_csv(f'{RESULTATS}/comparaison_finale_v3.csv')
print(f'Test : {len(test)} offres (1 offre = {100/len(test):.1f} points), étiquettes v3')
final

In [ ]:
# 11. Les sorties qui ont eu besoin d'une réparation
d = pd.read_csv(f'{RESULTATS}/details_finetune_v3_repare.csv')
for _, r in d[d['repare']].iterrows():
    print('URL :', r['url'])
    print('Réparé :', reparer(r['sortie_brute'])[:400])
    print('-' * 60)
if not d['repare'].any():
    print('Aucune réparation nécessaire : la v3 ne boucle pas sur le test.')

In [ ]:
# 12. Pousser l'adaptateur v3 sur Hugging Face (privé)
REPO = 'mayss1/qwen2.5-1.5b-offres-tn-lora'
model.push_to_hub(REPO, private=True, commit_message='v3 : etiquettes reannotees')
tokenizer.push_to_hub(REPO, private=True)
print('v3 poussée sur', REPO)

In [ ]:
v2 = pd.read_csv(f'{RESULTATS}/details_finetune_v2_rescore_v3.csv')
v3 = pd.read_csv(f'{RESULTATS}/details_finetune_v3_repare.csv')

def lire_pred(brut):
    try:
        return json.loads(nettoyer(brut))
    except json.JSONDecodeError:
        try:
            return json.loads(reparer(brut))
        except Exception:
            return {}

for champ in ['duree_mois', 'remuneration']:
    print(f'===== {champ} =====')
    for ex, (_, a), (_, b) in zip(test, v2.iterrows(), v3.iterrows()):
        if a[champ] != b[champ]:
            print('Attendu :', ex['extraction'][champ],
                  '| v2 :', lire_pred(a['sortie_brute']).get(champ),
                  '| v3 :', lire_pred(b['sortie_brute']).get(champ))
            print('  ', ex.get('url'))

In [ ]:
DUREE_RE = re.compile(
    r'\d+([.,]\d+)?\s*(mois|month|months|semaines?|weeks?|jours?|days?|ans?|years?)'
    r'|\d+\s*(يوما|يوم|أيام|أسابيع|أشهر|شهر)|يومين|شهرين',
    re.IGNORECASE)

def garde_fous(brut, offre):
    pred = lire_pred(brut)
    if not pred:
        return brut
    if pred.get('duree_mois') is not None and not DUREE_RE.search(offre):
        pred['duree_mois'] = None
    if isinstance(pred.get('remuneration'), str) and pred['remuneration'].strip().lower() in {'confidentiel', 'confidential'}:
        pred['remuneration'] = None
    return json.dumps(pred, ensure_ascii=False)

sorties_gf = [garde_fous(s, ex['offre']) for s, ex in zip(sorties, test)]
res_v3_gf = scorer('finetune_v3_gardefous', sorties_gf, reparation=True)

final = pd.concat([final, res_v3_gf.rename('v3 + garde-fous')], axis=1)
final.to_csv(f'{RESULTATS}/comparaison_finale_v3.csv')
final


In [ ]:
for url in ['stage-pfe-agent-ia-or-smart-production-assist',
            'stage-pfe-genie-logiciel-and-intelligence-artificielle']:
    ex = next(e for e in test if url in e['url'])
    print(url)
    for m in DUREE_RE.finditer(ex['offre']):
        print('   trouvé :', ex['offre'][max(0, m.start()-40):m.end()+20].replace('\n', ' '))

In [ ]:
import json, os, re, shutil, csv
from datetime import date
from google.colab import drive
drive.mount('/content/drive')

RACINE = '/content/drive/MyDrive/job-offer-extractor'
DOSSIER = f'{RACINE}/data'
FICHIERS = ['offres_propres.jsonl', 'train.jsonl', 'test.jsonl']
GUIDE = f'{RACINE}/GUIDE_ANNOTATION.md'
JOURNAL = f'{DOSSIER}/journal_duree_fourchettes.csv'

APPLIQUER = True

FOURCHETTE = re.compile(
    r'(\d+(?:[.,]\d+)?)\s*(?:-|–|à|a|to)\s*(\d+(?:[.,]\d+)?)\s*(mois|months?)\b',
    re.IGNORECASE)

def lire(chemin):
    with open(chemin, encoding='utf-8') as f:
        return [json.loads(l) for l in f if l.strip()]

def ecrire(chemin, lignes):
    with open(chemin, 'w', encoding='utf-8') as f:
        for l in lignes:
            f.write(json.dumps(l, ensure_ascii=False) + '\n')

changements = {}
for nom in FICHIERS:
    for ex in lire(f'{DOSSIER}/{nom}'):
        m = FOURCHETTE.search(ex['offre'])
        if not m:
            continue
        borne_basse = float(m.group(1).replace(',', '.'))
        actuel = ex['extraction']['duree_mois']
        url = ex.get('url')
        if actuel is None:
            changements[url] = (actuel, borne_basse, m.group(0))
        elif abs(actuel - borne_basse) > 0.05 :
           changements[url] = (actuel, borne_basse, m.group(0))

print(f'\n{len(changements)} offre(s) à corriger :')
for url, (avant, apres, extrait) in changements.items():
    print(f'  {avant} → {apres}   « {extrait} »   {url}')

if APPLIQUER and changements:
    sauvegarde = f'{DOSSIER}/sauvegarde_avant_fourchettes'
    os.makedirs(sauvegarde, exist_ok=True)
    for nom in FICHIERS:
        shutil.copy(f'{DOSSIER}/{nom}', f'{sauvegarde}/{nom}')
        lignes = lire(f'{DOSSIER}/{nom}')
        for ex in lignes:
            if ex.get('url') in changements:
                ex['extraction']['duree_mois'] = changements[ex['url']][1]
        ecrire(f'{DOSSIER}/{nom}', lignes)

    with open(JOURNAL, 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(['url', 'champ', 'avant', 'après', 'raison', 'date'])
        for url, (avant, apres, extrait) in changements.items():
            w.writerow([url, 'duree_mois', avant, apres,
                        f'fourchette « {extrait} » → borne basse', date.today().isoformat()])

    regle = ('\n\n## Durée\n\n'
             '- Durée en fourchette (« 3-6 mois », « 4 à 6 months ») → `duree_mois` = la borne basse (3).\n'
             '- Une date de publication ou d\'expiration (« Published 8 months ago ») n\'est jamais une durée.\n')
    with open(GUIDE, encoding='utf-8') as f:
        deja = '## Durée' in f.read()
    if not deja:
        with open(GUIDE, 'a', encoding='utf-8') as f:
            f.write(regle)

    print('\n✅ Corrections appliquées, sauvegarde dans', sauvegarde)
    print('✅ Journal :', JOURNAL)
    print('✅ Règle ajoutée au guide' if not deja else 'ℹ️ La règle existait déjà dans le guide')

In [ ]:
import matplotlib.pyplot as plt
from google.colab import drive
drive.mount('/content/drive')

RESULTATS = '/content/drive/MyDrive/job-offer-extractor/resultats'
etapes = [5, 10, 15, 20, 25, 30, 35, 40]
pertes = [0.475249, 0.305854, 0.202230, 0.169716, 0.138448, 0.121912, 0.078646, 0.114582]

plt.figure(figsize=(7, 4))
plt.plot(etapes, pertes, marker='o')
plt.xlabel('Étape'); plt.ylabel('Perte d’entraînement')
plt.title('Fine-tuning QLoRA v3 — Qwen2.5-1.5B (3 époques)')
plt.grid(alpha=0.3)
plt.savefig(f'{RESULTATS}/courbe_perte_v3.png', dpi=150, bbox_inches='tight')
plt.show()